In [1]:
import os

# 1. Ép Kaggle quay về thư mục gốc an toàn
os.chdir('/kaggle/working')

# 2. Xóa sạch folder RAG bị lỗi hoặc bị sót (nếu có)
!rm -rf RAG

# 3. Clone lại từ đầu (Thay link dưới đây bằng link Github của bạn)
!git clone https://github.com/NguyenngocLanUET/RAG

Cloning into 'RAG'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 27 (delta 11), reused 23 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 124.87 KiB | 2.66 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [2]:
import json
import re
import pandas as pd

def create_rag_evaluation_dataset(input_json_file, output_csv_file):
    print(f"Đang đọc dữ liệu từ: {input_json_file}")
    
    # 1. Đọc file JSON
    with open(input_json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    rag_test_data = []

    # 2. Duyệt qua từng bài viết
    for doc in data:
        title = doc.get("title", "")
        url = doc.get("url", "")
        # ĐÂY CHÍNH LÀ ĐOẠN VĂN SẠCH DÙNG CHO FAISS VÀ MODEL ĐỌC
        context = doc.get("rule_cleaned_content", "") 
        qa_text = doc.get("generated_qa", "")

        # 3. Dùng Regex để bóc tách từng cặp Câu hỏi (CH) và Câu trả lời (ĐA)
        pairs = re.findall(
            r"CH\d+:\s*(.*?)\s*\nĐA\d+:\s*(.*?)(?=\nCH\d+:|$)",
            qa_text + "\n",
            re.DOTALL
        )

        # 4. Ghi từng cặp QA thành 1 dòng riêng biệt
        for q, a in pairs:
            q = q.strip()
            a = a.strip()

            if q and a:
                rag_test_data.append({
                    "Question": q,             # Câu hỏi để đưa vào hàm tìm kiếm
                    "Ground_Truth": a,         # Đáp án chuẩn (để tính điểm Exact Match, F1)
                    "Context": context,        # Văn bản gốc (để đưa vào FAISS database)
                    "Title": title,            # Metadata (để biết nguồn)
                    "URL": url                 # Metadata (để biết nguồn)
                })

    # 5. Xuất ra file CSV
    df = pd.DataFrame(rag_test_data)
    df.to_csv(output_csv_file, index=False, encoding="utf-8-sig")
    
    print(f"✅ Đã tạo thành công file: {output_csv_file}")
    print(f"🎯 Tổng số câu hỏi test tạo được: {len(df)}")
    
    # In thử 1 dòng để kiểm tra
    if len(df) > 0:
        print("\n--- MẪU DỮ LIỆU ĐÃ BÓC TÁCH ---")
        print(f"Câu hỏi : {df.iloc[0]['Question']}")
        print(f"Đáp án  : {df.iloc[0]['Ground_Truth']}")
        print(f"Context : {df.iloc[0]['Context'][:100]}...")

# Chạy hàm (Thay tên file của bạn vào đây)
create_rag_evaluation_dataset("/kaggle/input/notebooks/nguyenthingoclanuet/generate-data/uet_qa_dataset.json", "RAG/rag_test_dataset.csv")

Đang đọc dữ liệu từ: /kaggle/input/notebooks/nguyenthingoclanuet/generate-data/uet_qa_dataset.json
✅ Đã tạo thành công file: RAG/rag_test_dataset.csv
🎯 Tổng số câu hỏi test tạo được: 399

--- MẪU DỮ LIỆU ĐÃ BÓC TÁCH ---
Câu hỏi : Trường Đại học Công nghệ tham gia tư vấn trực tuyến về chủ đề gì?
Đáp án  : Kỳ thi HAS và tuyển sinh khối ngành Công nghệ kỹ thuật và Khoa học Xã hội & Nhân văn.
Context : Ngày 12/3 tới đây, Trung tâm Khảo thí ĐHQGHN phối hợp với Trường ĐH Công nghệ và Trường ĐH Khoa học ...


In [3]:
!pip install -q faiss-cpu sentence-transformers langchain-text-splitters langchain-community transformers accelerate bitsandbytes pandas rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pan

In [4]:
# %%writefile /kaggle/working/RAG/build_index.py
# import os
# import json
# import faiss
# import numpy as np
# import pandas as pd
# from sentence_transformers import SentenceTransformer

# def build_pipeline(csv_path, output_dir):
#     os.makedirs(output_dir, exist_ok=True)
#     df = pd.read_csv(csv_path, encoding="utf-8-sig")
    
#     documents = []
#     bm25_corpus = []
    
#     print(f"Đang xử lý {len(df)} dòng dữ liệu...")
#     for idx, row in df.iterrows():
#         title = str(row.get("Title", "")).strip()
#         question = str(row.get("Question", "")).strip()
#         answer = str(row.get("Answer", "")).strip()
        
#         # QUAN TRỌNG: Tạo một câu khẳng định hoàn chỉnh từ QA
#         # Nếu Answer quá ngắn, việc gộp Question vào sẽ cứu vãn ngữ cảnh
#         full_context = f"Thông tin về {title}: Đối với câu hỏi '{question}', câu trả lời là '{answer}'."
        
#         doc_obj = {
#             "id": idx,
#             "text": full_context,
#             "metadata": {"answer": answer}
#         }
#         documents.append(doc_obj)
#         bm25_corpus.append(full_context.lower().split())

#     # Lưu dữ liệu
#     with open(f"{output_dir}/documents.json", "w", encoding="utf-8") as f:
#         json.dump(documents, f, ensure_ascii=False, indent=2)
#     with open(f"{output_dir}/bm25_corpus.json", "w", encoding="utf-8") as f:
#         json.dump(bm25_corpus, f, ensure_ascii=False)

#     # Tạo Index
#     model = SentenceTransformer("BAAI/bge-m3", device="cuda")
#     texts = [d["text"] for d in documents]
#     embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
    
#     index = faiss.IndexFlatIP(embeddings.shape[1])
#     index.add(np.array(embeddings).astype("float32"))
#     faiss.write_index(index, f"{output_dir}/index.faiss")
#     print("Xong!")

# if __name__ == "__main__":
#     build_pipeline("/kaggle/working/RAG/uet_qa_dataset_500_with_id.csv", "/kaggle/working/my_uet_index")

In [5]:
%%writefile /kaggle/working/RAG/build_index.py
import os
import json
import faiss
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

def process_documents(df):
    documents = []
    
    # RẤT QUAN TRỌNG: Lọc bỏ các Context trùng lặp
    # (Vì 1 bài báo/context có thể sinh ra nhiều câu hỏi trong file CSV)
    unique_df = df.drop_duplicates(subset=['Context']).reset_index(drop=True)
    
    print(f"--- Đã lọc ra {len(unique_df)} đoạn văn bản (Context) duy nhất để đưa vào Database ---")
    
    for idx, row in tqdm(unique_df.iterrows(), total=len(unique_df)):
        context = str(row.get("Context", "")).strip()
        title = str(row.get("Title", "")).strip()
        url = str(row.get("URL", "")).strip()

        # Bỏ qua các context rỗng hoặc quá ngắn
        if len(context) < 20:
            continue

        documents.append({
            "id": idx,
            "text": context, # Chỉ nhúng toàn bộ đoạn văn bản tự nhiên
            "metadata": {
                "title": title, 
                "url": url
            }
        })

    return documents

def build_pipeline(csv_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Đang đọc dữ liệu từ: {csv_path}")
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    
    # 1. Xử lý và gom nhóm Documents (Không cần LLM nữa)
    documents = process_documents(df)
    
    # 2. Lưu documents vào file json (để khi search ra còn lấy lại được text gốc)
    with open(f"{output_dir}/documents.json", "w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)

    # 3. Embedding với BGE-M3
    print("--- Đang khởi tạo mô hình BGE-M3 ---")
    embed_model = SentenceTransformer("BAAI/bge-m3", device="cuda")
    
    print("--- Đang tạo Vector Embeddings ---")
    texts = [d["text"] for d in documents]
    embeddings = embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
    
    # 4. Lưu vào FAISS Index
    print("--- Đang lưu trữ vào FAISS ---")
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(np.array(embeddings).astype("float32"))
    faiss.write_index(index, f"{output_dir}/index.faiss")
    
    print("✅ Hoàn tất xây dựng Index!")

if __name__ == "__main__":
    # LƯU Ý: Đổi tên file CSV dưới đây thành file CSV bạn vừa tạo ở bước trước (có cột Context)
    INPUT_CSV = "/kaggle/working/RAG/rag_test_dataset.csv" 
    OUTPUT_DIR = "/kaggle/working/my_uet_index"
    
    build_pipeline(INPUT_CSV, OUTPUT_DIR)

Overwriting /kaggle/working/RAG/build_index.py


In [6]:
%%writefile /kaggle/working/RAG/retriever.py
import json
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

class HybridRetriever:
    def __init__(self, bi_encoder_name="BAAI/bge-m3", reranker_name="BAAI/bge-reranker-base"):
        print("Loading BGE-M3 Bi-encoder...")
        self.bi_encoder = SentenceTransformer(bi_encoder_name, device="cuda")
        print("Loading BGE Reranker...")
        self.reranker = CrossEncoder(reranker_name, device="cuda")
        self.index = None
        self.documents = []
        self.bm25 = None

    def load(self, index_dir):
        print(f"Loading FAISS index from {index_dir}...")
        self.index = faiss.read_index(f"{index_dir}/index.faiss")
        
        print("Loading documents...")
        with open(f"{index_dir}/documents.json", "r", encoding="utf-8") as f:
            self.documents = json.load(f)
            
        # SỬA LỖI: Tạo BM25 trực tiếp từ danh sách documents vừa load
        print("Building BM25 corpus on the fly...")
        # Tách từ (tokenize) bằng cách lowercase và split() cho tiếng Việt cơ bản
        tokenized_corpus = [doc["text"].lower().split() for doc in self.documents]
        self.bm25 = BM25Okapi(tokenized_corpus)
        print("Retriever loaded successfully!")

    def search(self, query, top_k=3): # Nên để top_k = 3 cho Extractor đọc được nhiều ý
        # 1. BM25 (Bắt từ khóa chính xác)
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        # Lấy top 15 ứng viên từ BM25
        bm25_indices = np.argsort(bm25_scores)[-15:][::-1]

        # 2. Dense (Bắt ngữ nghĩa)
        q_emb = self.bi_encoder.encode([query], normalize_embeddings=True)
        # Lấy top 15 ứng viên từ Vector Dense
        _, dense_indices = self.index.search(np.array(q_emb).astype("float32"), 15)

        # Gộp ứng viên (Loại bỏ trùng lặp bằng set)
        all_indices = list(set(bm25_indices) | set(dense_indices[0]))
        candidates = [self.documents[i] for i in all_indices if i != -1]

        if not candidates:
            return []

        # 3. Rerank bằng BGE-Reranker
        pairs = [[query, c["text"]] for c in candidates]
        scores = self.reranker.predict(pairs)
        
        for i, s in enumerate(scores):
            candidates[i]["rerank_score"] = float(s)
        
        # Sắp xếp lại dựa trên điểm Reranker
        ranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
        
        # SỬA LỖI NGƯỠNG: Vì điểm là Logit, ta hạ ngưỡng xuống thấp (VD: -5.0) 
        # Hoặc tốt nhất là cứ trả về Top K, mô hình RoBERTa (Extractor) sẽ tự đánh giá score của nó.
        if ranked[0]["rerank_score"] < -3.0: 
            return []
            
        return ranked[:top_k]

Overwriting /kaggle/working/RAG/retriever.py


In [7]:
%%writefile /kaggle/working/RAG/generator.py
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

class QAGenerator:
    def __init__(self, model_id="Qwen/Qwen2.5-7B-Instruct"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
            device_map="auto"
        )

    def get_answer(self, question, retrieved_docs):
        if not retrieved_docs:
            return "Không có thông tin."

        # Kết hợp thông tin từ Top-K tài liệu
        combined_context = "\n---\n".join([f"Tài liệu {i+1}: {doc['text']}" for i, doc in enumerate(retrieved_docs)])
        
        # PROMPT bổ sung ví dụ thực tế (Few-shot) để định hình câu trả lời của mô hình 3B
        messages = [
            {"role": "system", "content": (
                "Bạn là trợ lý ảo chỉ trích xuất thông tin chính xác từ tài liệu được cung cấp.\n"
                "NHIỆM VỤ: Trả lời câu hỏi bằng cách trích xuất trực tiếp con số, mốc thời gian, hoặc tên thực thể.\n\n"
                "QUY TẮC BẮT BUỘC:\n"
                "1. CHỈ đưa ra đáp án trực tiếp. TUYỆT ĐỐI không viết thành câu đầy đủ, không lặp lại câu hỏi.\n"
                "2. Không thêm các từ thừa như 'Có', 'Là', 'Thời gian là', 'Hạn nộp là'.\n"
                "3. Trả lời dưới 10 từ.\n"
                "4. Nếu tài liệu không chứa câu trả lời, hãy trả lời: 'Không có thông tin'.\n\n"
                "VÍ DỤ MINH HỌA:\n"
                "- Câu hỏi: Có bao nhiêu ứng viên được triệu tập?\n"
                "  Trả lời SAI: Có 20 ứng viên được triệu tập tham dự thi.\n"
                "  Trả lời ĐÚNG: 20 người\n\n"
                "- Câu hỏi: Thời hạn nộp bản sao bằng tốt nghiệp là khi nào?\n"
                "  Trả lời SAI: Thời hạn nộp là trước thứ Sáu ngày 30/05.\n"
                "  Trả lời ĐÚNG: Trước thứ Sáu ngày 30/05"
            )},
            {"role": "user", "content": f"[TÀI LIỆU]:\n{combined_context}\n\n[CÂU HỎI]:\n{question}"}
        ]
        
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        with torch.inference_mode():
            # max_new_tokens giảm xuống còn 25 để ép mô hình ngắt câu sớm
            outputs = self.model.generate(
                **inputs, 
                max_new_tokens=25, 
                temperature=0.0, # Đặt bằng 0 để đảm bảo tính nhất quán cao nhất
                do_sample=False
            )
        
        answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1].strip()
        
        # Hậu xử lý cơ bản: Loại bỏ dấu chấm ở cuối câu trả lời nếu có (giúp tăng điểm EM)
        if answer.endswith("."):
            answer = answer[:-1].strip()
            
        return answer

Overwriting /kaggle/working/RAG/generator.py


In [8]:
import os
# Chạy build_index.py để tạo đủ 3 file: index.faiss, documents.json, bm25_corpus.json
!python /kaggle/working/RAG/build_index.py

Đang đọc dữ liệu từ: /kaggle/working/RAG/rag_test_dataset.csv
--- Đã lọc ra 200 đoạn văn bản (Context) duy nhất để đưa vào Database ---
100%|██████████████████████████████████████| 200/200 [00:00<00:00, 22668.24it/s]
--- Đang khởi tạo mô hình BGE-M3 ---
config_sentence_transformers.json: 100%|████████| 123/123 [00:00<00:00, 479kB/s]
README.md: 15.8kB [00:00, 38.1MB/s]
sentence_bert_config.json: 100%|██████████████| 54.0/54.0 [00:00<00:00, 283kB/s]
config.json: 100%|█████████████████████████████| 687/687 [00:00<00:00, 3.06MB/s]
pytorch_model.bin:  71%|██████████████▏     | 1.61G/2.27G [00:07<00:02, 229MB/s]
Loading weights: 100%|█| 391/391 [00:00<00:00, 2762.61it/s, Materializing param=
model.safetensors:   0%|                            | 0.00/2.27G [00:00<?, ?B/s]
tokenizer_config.json: 100%|███████████████████| 444/444 [00:00<00:00, 3.52MB/s]
model.safetensors:   0%|                            | 0.00/2.27G [00:00<?, ?B/s]
model.safetensors:   0%|                            | 0.00/2.2

In [9]:
import sys
import torch
import gc

# Dọn RAM
gc.collect()
torch.cuda.empty_cache()

sys.path.append('/kaggle/working/RAG')
from retriever import HybridRetriever
from generator import QAGenerator

# Load đúng thư mục chứa 3 file index
retriever = HybridRetriever()
retriever.load("/kaggle/working/my_uet_index") 

generator = QAGenerator()

query = "Trong kỳ tuyển dụng viên chức hành chính năm 2023 của Trường Đại học Công nghệ, Đại học Quốc gia Hà Nội, có bao nhiêu ứng viên được triệu tập tham dự thi Vòng 1?"
docs = retriever.search(query, top_k=1) # 500 câu ngắn thì chỉ cần top 2 là đủ
answer = generator.get_answer(query, docs)

print("\n" + "="*30)
print(f"CÂU HỎI: {query}")
print(f"ĐÁP ÁN: {answer}")
print("="*30)

Loading BGE-M3 Bi-encoder...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading BGE Reranker...


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



CÂU HỎI: Trong kỳ tuyển dụng viên chức hành chính năm 2023 của Trường Đại học Công nghệ, Đại học Quốc gia Hà Nội, có bao nhiêu ứng viên được triệu tập tham dự thi Vòng 1?
ĐÁP ÁN: 20 người


In [10]:
query = "Tuyển dụng viên chức hành chính năm 2023 của Trường Đại học Công nghệ, Đại học Quốc gia Hà Nội, có bao nhiêu người tham dự thi Vòng 1?"
docs = retriever.search(query, top_k=2)

print("--- DỮ LIỆU RETRIEVER TÌM ĐƯỢC ---")
for i, d in enumerate(docs):
    print(f"Đoạn {i+1}: {d['text']}")
    print(f"Rerank Score: {d['rerank_score']}")
    print("-" * 20)

--- DỮ LIỆU RETRIEVER TÌM ĐƯỢC ---
Đoạn 1: Thực hiện Thông báo số 54/TB-ĐHCN ngày 10 tháng 02 năm 2023 của Trường Đại học Công nghệ thông báo tuyển dụng viên chức hành chính, kỹ thuật năm 2023;
Căn cứ Kết quả thi Vòng 1 kỳ thi tuyển dụng viên chức hành chính, kỹ thuật Trường Đại học Công nghệ năm 2023;
Căn cứ Kết luận của Hội đồng tuyển dụng viên chức hành chính, kỹ thuật Trường Đại học Công nghệ năm 2023 tại cuộc họp ngày 23 tháng 3 năm 2023;
Hội đồng tuyển dụng viên chức hành chính, kỹ thuật Trường Đại học Công nghệ năm 2023 thông báo kết quả thi Vòng 1 và triệu tập ứng viên đủ điều kiện, tiêu chuẩn tham dự thi Vòng 2 kỳ thi tuyển dụng viên chức hành chính, kỹ thuật Trường Đại học Công nghệ năm 2023, cụ thể như sau:
1. Kết quả thi Vòng 1
– Số ứng viên được triệu tập tham dự thi Vòng 1 là: 20 người.
– Số ứng viên đủ điều kiện được tham dự thi Vòng 2 là: 20 người.
– Số ứng viên không đủ điều kiện tham dự thi Vòng 2 là: 0 người.
Kết quả Vòng 1 thi trắc nghiệm môn Kiến thức chung và môn 

In [11]:
import pandas as pd
df = pd.read_csv("/kaggle/working/RAG/uet_qa_dataset_500_with_id.csv")
search_term = "học phí"
results = df[df.apply(lambda row: row.astype(str).str.contains(search_term, case=False).any(), axis=1)]
print(f"Số câu chứa từ '{search_term}': {len(results)}")
if len(results) > 0:
    print(results[['Question', 'Answer']].head())

Số câu chứa từ 'học phí': 14
                                              Question  \
53   Trong kỳ tuyển dụng viên chức hành chính năm 2...   
54   Hiện nay đã hết thời gian gian nộp hồ sơ MGHP ...   
104  Quyết định số 2080/QĐ-ĐHCN ngày 29 tháng 09 nă...   
105  Trong học kỳ II năm học 2025-2026, Trường Đại ...   
146  Trong năm học 2024-2025, Trường Đại học Công n...   

                                                Answer  
53                                                 100  
54                                                 Có.  
104  Đây là quyết định của Hiệu trưởng Trường Đại h...  
105  Học phí đã được điều chỉnh cho các chương trìn...  
146                                                 Có  


In [12]:
%%writefile /kaggle/working/RAG/evaluate.py
import os
import re
import json
import string
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import Counter
from retriever import HybridRetriever
from generator import QAGenerator

# =========================================================
# UTILS: CHUẨN HÓA VĂN BẢN
# =========================================================
def normalize_answer(s):
    """Lấy chữ thường, bỏ dấu câu, bỏ khoảng trắng thừa"""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_punc(lower(str(s))))

# =========================================================
# METRICS: EM, F1, PRECISION, RECALL (TOKEN LEVEL)
# =========================================================
def calc_f1_precision_recall(prediction, ground_truth):
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    
    if len(prediction_tokens) == 0 or len(ground_truth_tokens) == 0:
        return int(prediction_tokens == ground_truth_tokens), 0, 0, 0

    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0, 0, 0, 0
    
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    
    return int(normalize_answer(prediction) == normalize_answer(ground_truth)), f1, precision, recall

# =========================================================
# MAIN EVALUATION
# =========================================================
def run_evaluation(csv_path, index_dir, num_samples=100):
    print("--- ĐANG KHỞI TẠO HỆ THỐNG RAG ---")
    retriever = HybridRetriever()
    retriever.load(index_dir)
    generator = QAGenerator()

    # Load dữ liệu test
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    if num_samples < len(df):
        df = df.sample(num_samples, random_state=42)

    results = []
    
    print(f"--- ĐANG ĐÁNH GIÁ TRÊN {len(df)} MẪU ---")
    for _, row in tqdm(df.iterrows(), total=len(df)):
        question = row['Question']
        ground_truth = row['Answer']
        
        # 1. Retrieval
        retrieved_docs = retriever.search(question, top_k=3)
        context_text = " ".join([d['text'] for d in retrieved_docs])
        
        # Kiểm tra xem Ground Truth có xuất hiện trong Context không (Retrieval Recall)
        retrieval_success = 1 if normalize_answer(ground_truth) in normalize_answer(context_text) else 0
        
        # 2. Generation
        try:
            prediction = generator.get_answer(question, retrieved_docs)
        except Exception as e:
            print(f"Lỗi khi generate: {e}")
            prediction = ""

        # 3. Calculate Metrics
        em, f1, prec, rec = calc_f1_precision_recall(prediction, ground_truth)
        
        results.append({
            "Question": question,
            "Ground Truth": ground_truth,
            "Prediction": prediction,
            "EM": em,
            "F1": f1,
            "Precision": prec,
            "Recall": rec,
            "Retrieval_Success": retrieval_success
        })

        # Giải phóng bộ nhớ GPU tránh OOM trên Kaggle
        torch.cuda.empty_cache()

    # Tạo DataFrame kết quả
    res_df = pd.DataFrame(results)
    
    # Tính trung bình
    print("\n" + "="*40)
    print("KẾT QUẢ ĐÁNH GIÁ CHI TIẾT")
    print("="*40)
    print(f"Exact Match (EM):      {res_df['EM'].mean():.4f}")
    print(f"F1-Score:              {res_df['F1'].mean():.4f}")
    print(f"Precision:             {res_df['Precision'].mean():.4f}")
    print(f"Recall (Generation):   {res_df['Recall'].mean():.4f}")
    print(f"Retrieval Recall@3:    {res_df['Retrieval_Success'].mean():.4f}")
    print("="*40)
    
    # Lưu file report
    report_path = "/kaggle/working/evaluation_report.csv"
    res_df.to_csv(report_path, index=False, encoding="utf-8-sig")
    print(f"Đã lưu báo cáo chi tiết vào: {report_path}")

if __name__ == "__main__":
    CSV_TEST = "/kaggle/working/RAG/uet_qa_dataset_500_with_id.csv"
    INDEX_DIR = "/kaggle/working/my_uet_index"
    
    # Bạn có thể đổi num_samples thành len(df) để test hết 500 câu
    run_evaluation(CSV_TEST, INDEX_DIR, num_samples=50)

Overwriting /kaggle/working/RAG/evaluate.py


In [13]:
!export PYTHONPATH=$PYTHONPATH:/kaggle/working/RAG && python /kaggle/working/RAG/evaluate.py

--- ĐANG KHỞI TẠO HỆ THỐNG RAG ---
Loading BGE-M3 Bi-encoder...
Loading weights: 100%|█| 391/391 [00:00<00:00, 2475.15it/s, Materializing param=
Loading BGE Reranker...
Loading weights: 100%|█| 201/201 [00:00<00:00, 526.58it/s, Materializing param=r
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading FAISS index from /kaggle/working/my_uet_index...
Loading documents...
Building BM25 corpus on the fly...
Retriever loaded successfully!
Loading weights:  76%|▊| 256/339 [00:22<00:07, 11.19it/s, Materializing param=mo
Traceback (most recent call last):
  File "/kaggle/working/RAG/evaluate.py", line 131, in <module>
    run_evaluation(CSV_TEST, INDEX_DIR, num_samples=50)
  File "/kaggle/wor